# Taxi Demand Prediction - Học Máy Thống Kê (DS102)
## Mở đầu
Notebook này xây dựng mô hình dự đoán nhu cầu gọi xe taxi dựa trên các yếu tố như giờ trong ngày, khu vực, dự báo thời tiết và các sự kiện đặc biệt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Thiết lập giao diện biểu đồ cơ bản
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Sinh dữ liệu mô phỏng theo quy luật (Data Generation)
Chúng ta sẽ tự động sinh ra khoảng 10,000 dữ liệu dựa trên quy luật:
- **Giờ cao điểm:** 7-9h sáng & 17-19h chiều thì khách đông.
- **Khu vực:** Khu vực A (Trung tâm) đông gấp hai lần các vùng khác.
- **Mưa & Sự kiện:** Làm nhu cầu gọi xe qua ứng dụng tăng mạnh.

In [ ]:
np.random.seed(42)
n_samples = 10000

# Sinh các Features đầu vào
hour = np.random.randint(0, 24, n_samples)
day_of_week = np.random.randint(0, 7, n_samples)
zone = np.random.choice(['A', 'B', 'C'], n_samples)
is_rain = np.random.choice([0, 1], n_samples, p=[0.8, 0.2])
is_event = np.random.choice([0, 1], n_samples, p=[0.95, 0.05])

# Sinh dữ liệu Label (Demand) dựa trên trọng số
base_demand = np.random.normal(15, 3, n_samples)

# Tính chỉ số nhân cho các điều kiện
peak_hour_mult = np.where(((hour >= 7) & (hour <= 9)) | ((hour >= 17) & (hour <= 19)), 2.5, 1.0)
zone_mult = np.where(zone == 'A', 2.0, np.where(zone == 'C', 1.5, 0.8))
rain_mult = np.where(is_rain == 1, 1.3, 1.0)
event_mult = np.where(is_event == 1, 3.0, 1.0)

# Tính toán Demand cuối cùng
demand = base_demand * peak_hour_mult * zone_mult * rain_mult * event_mult
demand = np.maximum(0, np.round(demand)).astype(int) # Demand luôn lớn hơn bằng 0 và nguyên

# Ghép thành DataFrame
df = pd.DataFrame({
    'hour': hour,
    'day_of_week': day_of_week,
    'zone': zone,
    'is_rain': is_rain,
    'is_event': is_event,
    'demand': demand
})

print(df.head())

## 2. Tiền xử lý dữ liệu và Khám phá dữ liệu (EDA)
EDA làm cơ sở để chứng minh dữ liệu phản ánh hiện thực.

In [ ]:
# Biểu đồ 1: Sự phân bố của tổng lượng yêu cầu
plt.figure(figsize=(10, 5))
sns.histplot(df['demand'], bins=40, kde=True, color='teal')
plt.title('Phân phối số chuyến xe (Taxi Demand)')
plt.xlabel('Số liệu Demand')
plt.ylabel('Tần suất xuất hiện')
plt.show()

# Biểu đồ 2: So sánh Demand trung bình theo các giờ
plt.figure(figsize=(12, 5))
sns.barplot(data=df, x='hour', y='demand', errorbar=None, palette='viridis')
plt.title('Sự chênh lệch Demand theo từng giờ trong ngày (Lưu ý 7-9h và 17-19h)')
plt.xlabel('Giờ trong ngày')
plt.show()

In [ ]:
# Chuyển đối biến Zone thành One-hot Encoding
df_encoded = pd.get_dummies(df, columns=['zone'], drop_first=True)
X = df_encoded.drop('demand', axis=1)
y = df_encoded['demand']

# Chia tập Train/Test = 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Kích thước Train: {X_train.shape}")
print(f"Kích thước Test: {X_test.shape}")

## 3. Huấn luyện mô hình Học máy Thống kê
- Mô hình **Linear Regression**: Hoạt động đơn giản, đo lường mối quan hệ tuyến tính.
- Mô hình **Poisson Regression**: Đặc thù tốt nhất cho Regression với giá trị đếm số chuyến.
- Mô hình **Random Forest**: Thêm vào để tham khảo đánh giá độ hoàn thiện.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Poisson Regression": PoissonRegressor(max_iter=1000),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    
    # Đảm bảo số chuyến đoán ra không âm
    y_pred = np.maximum(0, np.round(model.predict(X_test)))
    
    # Tính metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results[name] = {'MAE': mae, 'RMSE': rmse}

results_df = pd.DataFrame(results).T
display(results_df)

## 4. Đánh giá Tổng thể
Biểu diễn trực quan sai số để so sánh.

In [ ]:
results_df.plot(kind='bar', figsize=(10, 5), colormap='Set2')
plt.title('So sánh sai số (MAE & RMSE) giữa 3 mô hình học máy')
plt.ylabel('Sai số đánh giá (Chuyến)')
plt.xticks(rotation=0)
plt.show()